In [1]:
# cell 1
# Mount Google Drive and set project workspace.

import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_final_project_dir():
    candidates = [
        "/content/drive/MyDrive/final_project",
        "/content/drive/MyDrive/final_project/",
    ]

    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project"):
                return os.path.abspath(root)

    raise FileNotFoundError("Could not find final_project in Drive.")

PROJECT_DIR = find_final_project_dir()
os.chdir(PROJECT_DIR)

print("PROJECT_DIR =", PROJECT_DIR)
print("CWD =", os.getcwd())

Mounted at /content/drive
PROJECT_DIR = /content/drive/MyDrive/final_project
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project


In [2]:
# cell 2
# Install evaluation and sentence splitting dependencies.

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "pip",
    "setuptools",
    "wheel",
])

pkgs = [
    "sentence-transformers",
    "tqdm==4.66.2",
    "pandas",
    "blingfire",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
] + pkgs)

print("Installed OK")

Installed OK


In [3]:
# cell 3
# Use CUDA for Colab GPU.

import torch

assert torch.cuda.is_available(), "GPU is required. Please enable GPU in Colab."

DEVICE = "cuda"

print("DEVICE =", DEVICE)
print("GPU =", torch.cuda.get_device_name(0))
print("GPU memory GB =", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

DEVICE = cuda
GPU = NVIDIA L4
GPU memory GB = 22.03


In [4]:
# cell 4
# Define BM25 evidence paths for both datasets.

import os

BM25_EVIDENCE_DIR = os.path.join(
    PROJECT_DIR,
    "BM25",
    "evidence",
)

TWOWIKI_EVIDENCE_PATH = os.path.join(
    BM25_EVIDENCE_DIR,
    "2wikimultihopqa_evidence.json",
)

HOTPOTQA_EVIDENCE_PATH = os.path.join(
    BM25_EVIDENCE_DIR,
    "hotpotqa_evidence.json",
)

DATASET_CONFIGS = {
    "2wikimultihopqa": {
        "name": "2wikimultihopqa",
        "path": TWOWIKI_EVIDENCE_PATH,
        "expected_types": None,
    },
    "hotpotqa": {
        "name": "hotpotqa",
        "path": HOTPOTQA_EVIDENCE_PATH,
        "expected_types": None,
    },
}

for dataset_name, config in DATASET_CONFIGS.items():
    print(dataset_name, "=>", config["path"])
    assert os.path.isfile(config["path"]), f"Missing evidence file: {config['path']}"

print("Both BM25 evidence files exist.")

2wikimultihopqa => /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json
hotpotqa => /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json
Both BM25 evidence files exist.


In [5]:
# cell 5
# Load the same embedding model used in the GitHub code.

import json
import pandas as pd

from tqdm import tqdm
from collections import Counter, defaultdict
from sentence_transformers import util
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
THRESHOLD = 0.9

emb = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Threshold:", THRESHOLD)
print("Device:", DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Threshold: 0.9
Device: cuda


In [6]:
# cell 6
# Load both JSON files.

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

DATASETS = {}

for dataset_name, config in DATASET_CONFIGS.items():
    DATASETS[dataset_name] = load_json(config["path"])

    print("JSON loaded:", dataset_name)
    print("Records:", len(DATASETS[dataset_name]))
    print("-" * 80)

JSON loaded: 2wikimultihopqa
Records: 1000
--------------------------------------------------------------------------------
JSON loaded: hotpotqa
Records: 1000
--------------------------------------------------------------------------------


In [7]:
# cell 7
# Prepare sentence splitter.

from blingfire import text_to_sentences

def split_text_to_sentences(text):
    if text is None:
        return []

    text = str(text).replace("\n", " ").strip()
    if not text:
        return []

    sentences = text_to_sentences(text).split("\n")

    sentences = [
        s.strip()
        for s in sentences
        if s and s.strip()
    ]

    return sentences

print("Sentence splitter ready.")

Sentence splitter ready.


In [8]:
# cell 8
# Convert evidence chunks into title-prefixed evidence sentences.

def evidence_chunks_to_sentences(record, add_title=True):
    evidence_chunks = record.get("evidence_chunk", [])

    found_evidence = []
    seen = set()

    for chunk in evidence_chunks:
        title = str(chunk.get("title", "")).strip()
        text = str(chunk.get("text", "")).strip()

        sentences = split_text_to_sentences(text)

        for sent in sentences:
            if add_title and title:
                evidence_sentence = f"Title: {title}. Evidence: {sent}"
            else:
                evidence_sentence = sent

            if evidence_sentence not in seen:
                found_evidence.append(evidence_sentence)
                seen.add(evidence_sentence)

    return found_evidence

print("Evidence chunk converter ready.")

Evidence chunk converter ready.


In [ ]:
# cell 9
# Inspect dataset size, question types, and converted evidence sentences.

for dataset_name, data in DATASETS.items():
    print("=" * 80)
    print("DATASET:", dataset_name)
    print("Path:", DATASET_CONFIGS[dataset_name]["path"])
    print("Records:", len(data))
    print("Type counts:", Counter(record["type"] for record in data))
    print("First keys:", list(data[0].keys()))
    print("First question:", data[0]["question"])
    print("First answer:", data[0]["answer"])
    print("First supports:", len(data[0]["supports"]))
    print("First evidence chunks:", len(data[0].get("evidence_chunk", [])))

    first_sentences = evidence_chunks_to_sentences(data[0], add_title=True)

    print("First converted evidence sentences:", len(first_sentences))
    print("\nSample converted evidence sentences:")
    for s in first_sentences[:10]:
        print("-", s)

    print("\n")

DATASET: 2wikimultihopqa
Path: /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json
Records: 1000
Type counts: Counter({'bridge_comparison': 250, 'compositional': 250, 'comparison': 250, 'inference': 250})
First keys: ['type', 'question', 'answer', 'supports', 'evidence_chunk']
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First answer: Kamakalawa
First supports: 4
First evidence chunks: 4
First converted evidence sentences: 30

Sample converted evidence sentences:
- Title: John Jaffer Janardhanan. Evidence: John Jaffer Janardhanan is a 1982 Malayalam movie directed by I. V. Sasi, written by T. Damodaran, starring Ratheesh, Ravindran and Mammootty.
- Title: John Jaffer Janardhanan. Evidence: It is a remake of the 1977 Hindi movie Amar Akbar Anthony.
- Title: John Jaffer Janardhanan. Evidence: Mammootty as 'John, Jaffer, Janardhanan' Soundtrack: The music was composed by Shyam and the lyrics were written by Sr

In [9]:
# cell 10
# GitHub-style support-level EM over sentence-split evidence chunks.

def evaluate_em_github_style(data, dataset_name, threshold=0.9):
    total_supports = 0
    correct_supports = 0

    total_by_type = defaultdict(int)
    correct_by_type = defaultdict(int)

    empty_evidence_records = 0

    for record in tqdm(data, total=len(data), desc=f"Evaluating {dataset_name}"):
        q_type = record["type"]
        supports = record["supports"]

        found_evidence = evidence_chunks_to_sentences(
            record=record,
            add_title=True,
        )

        if len(found_evidence) == 0:
            empty_evidence_records += 1
            evidence_emb = None
        else:
            evidence_emb = emb.encode(
                found_evidence,
                device=DEVICE,
            )

        for s in supports:
            total_supports += 1
            total_by_type[q_type] += 1

            if evidence_emb is None:
                continue

            support_emb = emb.encode(
                s[1],
                device=DEVICE,
            )

            sim_score = util.dot_score(
                support_emb,
                evidence_emb,
            ).cpu().numpy().flatten()

            if max(sim_score) > threshold:
                correct_supports += 1
                correct_by_type[q_type] += 1

    overall_em = correct_supports / total_supports if total_supports else 0.0

    results = {
        "overall": {
            "total_supports": total_supports,
            "correct_supports": correct_supports,
            "em": overall_em,
        },
        "by_type": {},
        "diagnostics": {
            "empty_evidence_records": empty_evidence_records,
        },
    }

    for q_type in sorted(total_by_type.keys()):
        total = total_by_type[q_type]
        correct = correct_by_type[q_type]
        em = correct / total if total else 0.0

        results["by_type"][q_type] = {
            "total_supports": total,
            "correct_supports": correct,
            "em": em,
        }

    return results

print("GitHub-style EM evaluator ready.")

GitHub-style EM evaluator ready.


In [10]:
# cell 11
# Run EM evaluation and show results for 2WikiMultiHopQA.

DATASET_NAME = "2wikimultihopqa"
data = DATASETS[DATASET_NAME]

print("\n" + "=" * 100)
print("DATASET:", DATASET_NAME)
print("=" * 100)

results_2wikimultihopqa = evaluate_em_github_style(
    data=data,
    dataset_name=DATASET_NAME,
    threshold=THRESHOLD,
)

overall = results_2wikimultihopqa["overall"]

print("\nOverall EM:")
print(
    f"Total supports: {overall['total_supports']} | "
    f"Correct supports: {overall['correct_supports']} | "
    f"EM: {overall['em']:.4f}"
)

print("\nEM by question type:")
for q_type, type_result in results_2wikimultihopqa["by_type"].items():
    print(
        f"{q_type} | "
        f"Total supports: {type_result['total_supports']} | "
        f"Correct supports: {type_result['correct_supports']} | "
        f"EM: {type_result['em']:.4f}"
    )

print("\nDiagnostics:")
print("Empty evidence records:", results_2wikimultihopqa["diagnostics"]["empty_evidence_records"])


all_rows = []

all_rows.append({
    "dataset": DATASET_NAME,
    "split": "overall",
    "total_supports": overall["total_supports"],
    "correct_supports": overall["correct_supports"],
    "em": overall["em"],
})

for q_type, type_result in results_2wikimultihopqa["by_type"].items():
    all_rows.append({
        "dataset": DATASET_NAME,
        "split": q_type,
        "total_supports": type_result["total_supports"],
        "correct_supports": type_result["correct_supports"],
        "em": type_result["em"],
    })

em_df_2wikimultihopqa = pd.DataFrame(all_rows)

em_df_2wikimultihopqa["em"] = em_df_2wikimultihopqa["em"].round(4)

em_df_2wikimultihopqa = em_df_2wikimultihopqa.sort_values(
    by=["dataset", "split"],
    key=lambda col: col.map(lambda x: "000_overall" if x == "overall" else str(x))
    if col.name == "split"
    else col,
).reset_index(drop=True)

display(em_df_2wikimultihopqa)


print("\n" + "#" * 100)
print("DATASET:", DATASET_NAME)
print("#" * 100)

for _, row in em_df_2wikimultihopqa.iterrows():
    print(
        f"{row['split']}: "
        f"Total supports: {int(row['total_supports'])} | "
        f"Correct supports: {int(row['correct_supports'])} | "
        f"EM: {row['em']:.4f}"
    )


DATASET: 2wikimultihopqa


Evaluating 2wikimultihopqa: 100%|██████████| 1000/1000 [00:42<00:00, 23.54it/s]


Overall EM:
Total supports: 2501 | Correct supports: 1052 | EM: 0.4206

EM by question type:
bridge_comparison | Total supports: 1000 | Correct supports: 399 | EM: 0.3990
comparison | Total supports: 501 | Correct supports: 382 | EM: 0.7625
compositional | Total supports: 500 | Correct supports: 151 | EM: 0.3020
inference | Total supports: 500 | Correct supports: 120 | EM: 0.2400

Diagnostics:
Empty evidence records: 0


,dataset,split,total_supports,correct_supports,em
0,2wikimultihopqa,overall,2501,1052,0.4206
1,2wikimultihopqa,bridge_comparison,1000,399,0.3990
2,2wikimultihopqa,comparison,501,382,0.7625
3,2wikimultihopqa,compositional,500,151,0.3020
4,2wikimultihopqa,inference,500,120,0.2400



####################################################################################################
DATASET: 2wikimultihopqa
####################################################################################################
overall: Total supports: 2501 | Correct supports: 1052 | EM: 0.4206
bridge_comparison: Total supports: 1000 | Correct supports: 399 | EM: 0.3990
comparison: Total supports: 501 | Correct supports: 382 | EM: 0.7625
compositional: Total supports: 500 | Correct supports: 151 | EM: 0.3020
inference: Total supports: 500 | Correct supports: 120 | EM: 0.2400


In [11]:
# cell 12
# Run EM evaluation and show results for HotpotQA.

DATASET_NAME = "hotpotqa"
data = DATASETS[DATASET_NAME]

print("\n" + "=" * 100)
print("DATASET:", DATASET_NAME)
print("=" * 100)

results_hotpotqa = evaluate_em_github_style(
    data=data,
    dataset_name=DATASET_NAME,
    threshold=THRESHOLD,
)

overall = results_hotpotqa["overall"]

print("\nOverall EM:")
print(
    f"Total supports: {overall['total_supports']} | "
    f"Correct supports: {overall['correct_supports']} | "
    f"EM: {overall['em']:.4f}"
)

print("\nEM by question type:")
for q_type, type_result in results_hotpotqa["by_type"].items():
    print(
        f"{q_type} | "
        f"Total supports: {type_result['total_supports']} | "
        f"Correct supports: {type_result['correct_supports']} | "
        f"EM: {type_result['em']:.4f}"
    )

print("\nDiagnostics:")
print("Empty evidence records:", results_hotpotqa["diagnostics"]["empty_evidence_records"])


all_rows = []

all_rows.append({
    "dataset": DATASET_NAME,
    "split": "overall",
    "total_supports": overall["total_supports"],
    "correct_supports": overall["correct_supports"],
    "em": overall["em"],
})

for q_type, type_result in results_hotpotqa["by_type"].items():
    all_rows.append({
        "dataset": DATASET_NAME,
        "split": q_type,
        "total_supports": type_result["total_supports"],
        "correct_supports": type_result["correct_supports"],
        "em": type_result["em"],
    })

em_df_hotpotqa = pd.DataFrame(all_rows)

em_df_hotpotqa["em"] = em_df_hotpotqa["em"].round(4)

em_df_hotpotqa = em_df_hotpotqa.sort_values(
    by=["dataset", "split"],
    key=lambda col: col.map(lambda x: "000_overall" if x == "overall" else str(x))
    if col.name == "split"
    else col,
).reset_index(drop=True)

display(em_df_hotpotqa)


print("\n" + "#" * 100)
print("DATASET:", DATASET_NAME)
print("#" * 100)

for _, row in em_df_hotpotqa.iterrows():
    print(
        f"{row['split']}: "
        f"Total supports: {int(row['total_supports'])} | "
        f"Correct supports: {int(row['correct_supports'])} | "
        f"EM: {row['em']:.4f}"
    )


DATASET: hotpotqa


Evaluating hotpotqa: 100%|██████████| 1000/1000 [00:38<00:00, 25.70it/s]


Overall EM:
Total supports: 2400 | Correct supports: 945 | EM: 0.3937

EM by question type:
bridge | Total supports: 1724 | Correct supports: 591 | EM: 0.3428
comparison | Total supports: 676 | Correct supports: 354 | EM: 0.5237

Diagnostics:
Empty evidence records: 0


,dataset,split,total_supports,correct_supports,em
0,hotpotqa,overall,2400,945,0.3938
1,hotpotqa,bridge,1724,591,0.3428
2,hotpotqa,comparison,676,354,0.5237



####################################################################################################
DATASET: hotpotqa
####################################################################################################
overall: Total supports: 2400 | Correct supports: 945 | EM: 0.3938
bridge: Total supports: 1724 | Correct supports: 591 | EM: 0.3428
comparison: Total supports: 676 | Correct supports: 354 | EM: 0.5237
